# Summary of Chemical Data Cleaning Work


The `DataCleaning.ipynb` notebook cleans the two raw and dirty chemical datasets,
`1991Che_DRSites.xls` and `log21P_MarjorChe_Sites.xlsx` stored in the `two_raw_datasets` folder. 

They contain the whole available chemical data, but are not in consistent formats and having some hidden issues within them.

The cleaning process is listed with these targeted issues in the following steps:

### Clean the log21 major chemical dataset

**Input**: the originally raw and dirty `log21P_MarjorChe_Sites.xlsx` dataset.

**Process**: 

Remove the rows with all missings and Retain needed columns $\rightarrow$ 

Back-transform the log2(Y+1) to the original Y values $\rightarrow$

Infer the units of all 16 chemicals $\rightarrow$ 

Internally uniform the units of Hg and Zn values $\rightarrow$

Record units.

**Output**: a cleaned chemical data set of shape (245, 22), ready for merging.

**Details**:

Specifically using the third sheet in the `log21P_MarjorChe_Sites.xlsx` file, which contains the chemical data mainly collected in 1999, 2004 and 2005.

1. Remove the rows with all missings in all chemical columns, which are the rows with year label - 1991.
Retain only the needed 16 chemical columns with site IDs and relevant information.

1. Back-transform the log2(Y+1) values to the original Y values.
   
2. Infer the units of these chemicals in Y values.
    
    Reasons:
   - There were no attached units in the original dataset.
   - These chemicals were apparently in different units as some chemicals are trivial elements needing smaller scales, while some chemicals are compounds needing larger scales.

3. Inspect the **Hg** and **Zn** columns and internally uniform the units of values within them. 
   
   That is: the 2004/2005 Hg values in **ug/g** are $\rightarrow$ **ng/g**; the 1999 Zn values in **mg/g** are $\rightarrow$ **ug/g**.

   $$\text{Hg}_{ug/g} \times 1000 = \text{Hg}_{ng/g}$$
    $$\text{Zn}_{mg/g} \times 1000 = \text{Zn}_{ug/g}$$
   
   Reasons:
   - The Hg values collected in 2004 and 2005 were in **ug/g** unit, while the rest of the Hg values were in **ng/g** unit. These 2004 and 2005 values are apparently **smaller** than others and perfectly related with the year labels.
 
   - The Zn values collected in 2004 and 2005 were in **ug/g** unit, while the rest values were in **mg/g**. These 2004 and 2005 values are much larger than others and perfectly related with the year labels.

**Conclusion**: After the steps from 0 to 3, the original ``log21P_MarjorChe_Sites.xlsx`` dataset is cleaned with 245 rows of complete chemical data in internally uniformed units. Not all the chemicals are in same units, some are in ug/g and other are in ng/g, this needs to be operated when merging with the `1991Che_DRSites.xlx` dataset.

### Clean the 1991 chemical dataset

**Input**: the originally raw and dirty `1991Che_DRSites.xls` dataset.

**Process**: Retain needed columns $\rightarrow$ Fill missings $\rightarrow$ Record units.

**Output**: the cleaned 1991 chemical data of shape (77, 22) ready for merging.

**Details**:

0. Retain the same 16 targeted chemical columns with site IDs and relevant information.

1. Fill the missing values in Hg(1), SumPCBs(45), Cd(2), OCS(63) and p,p'-DDE(52).
Apply the same filling approach with unifom random values draw from interval [0.01X, 0.5X], where X is the detection limit of each chemical. The detection limits X recorded in the `1991Che_DRSites.xls` data were in consistent units as their reponsible chemicals, not uniformly but internally consistent.

    Reasons:
    - There were missing values in this dataset, specifically 1 missing in Hg, 45 missings in SumPCBs, 2 missings in Cd, 63 missings in OCS and 52 missings in p,p'-DDE. 
    - The filling approach is based on Jian's procedure in 2008, page 31.

2. Record the units of these chemicals in the cleaned 1991 DR dataset.
   
   Reasons:
      - The unit array of the 16 chemicals in this 1991 dataset is not same as the unit array of the 16 chemicals in the cleaned log21 major chemical dataset.
      - In order to merge these two cleaned datasets, their units need to be compared first and then transform them into a uniformed unit array.

**Conclusions**: As the units scales were already clean in the original 1991 dataset, not much inspection work was needed. Retaining the necessary chemical columns and filling the missings makes the 1991 dataset clean and ready for merging. The last and must thing is to record the units of these chemicals in the cleaned 1991 dataset, which will and must guide how to merge it with the cleaned log21 major chemical dataset.

### Merge the two cleaned datasets for a clean and complete chemical dataset

**Inputs**: the cleaned log21 major chemical dataset and the cleaned 1991 DR chemical dataset.

**Process**:

Compare and uniform the units of the two data sets, elementwise separately $\rightarrow$

Scan the cleaned log21 major chemical dataset, drop off the duplicated rows that can be found in the cleaned 1991 dataset $\rightarrow$

Concatenate the two datasets $\rightarrow$

Record the units of the merged dataset

**Outputs**:

A complete and row-wise unique chemical dataset of shape (322, 22), ready for use.

**Details**:

1. Compare the units of the two cleaned datasets, elementwise, use a target unit array to uniform their units.

    Reasons:
    - As did discussed before, the unit arrays of the two cleaned datasets are not same, merging them needs to 
    uniform their units by a universal unit array.

2. Scan the cleaned log21 major chemical dataset, drop off the duplicated rows that can be found in the cleaned 1991 dataset. It resulted 233 sites collected in 1999, 2004 and 2005 in the deduplicated log21 major chemical dataset.

    Reasons:
    - The log21 major chemical dataset contains samples collected across 1991, 1999, 2004 and 2005, but only the 1991 samples are not complete (not having missing but there were more samples collected in 1991).
  
    - The 1991 DR chemical dataset contains all samples collected in 1991, which are complete and therefore
    overlap with the 1991 samples in the log21 major chemical dataset.

2. Concatenate the two datasets.

    Reasons:
    - It is not riching more information around sites by merging data in sample-wise,
    but it is stacking more the same type of information with more samples.
    - Therefore, no common key is needed for concatenating the two datasets, only the same column shape and names are needed.

3. Record the units of the merged dataset.

    Reasons:
    - These units are the first-hand numerical scales of them, reflect the interpretation and 
    almost determine the **effects of transformations** that are applied right before modeling.

**Conclusions**:
After the steps from 0 to 3, the two datasets were stacked together with a total 322 rows of complete chemical data,
16 chemical columns and 6 columns of site IDs and relevant information (322, 16 + 6).

The finalized universal unit array that transformed the two distinct unit arrays is listed below:

<div align="center">

<table style="font-size: 12px; text-align: center;">
  <thead>
    <tr>
      <th>Chemical</th>
      <th>major_set_unit</th>
      <th>1991_DR_unit</th>
      <th>target_unit</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Co</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Al</td><td>ug/g</td><td>mg/g</td><td>ug/g</td>
    </tr>
    <tr><td>Ni</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Mn</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
    <td>Fe</td><td>ug/g</td><td>mg/g</td><td>ug/g</td>
    </tr>
    <tr><td>Cr</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Cu</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Hg</td><td>ng/g or ug/g</td><td>ug/g</td><td>ng/g</td>
    </tr>
    <tr><td>Pb</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Zn</td><td>ug/g or mg/g</td><td>ug/g</td><td>ug/g</td>
    </tr>
    <tr><td>SumPCBs</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>Cd</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>OCS</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>p,p'-DDE</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>As</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;"> 
      <td>Ca</td><td>ug/g</td><td>mg/g</td><td>ug/g</td>
    </tr>
  </tbody>
</table>

</div>

Notes:
1. Chemicals with different units in the two datasets are highlighted in yellow.
   
2. The Hg and Zn values were in mixed units in the original log21 major chemical dataset, they were uniformed into one consistent unit (ng/g for Hg and ug/g for Zn) first, then converted to target units.

3. The equivalence between units:
   $$
   1 \text{ mg/g} = 1000 \text{ ug/g}; \quad 1 \text{ ug/g} = 1000 \text{ ng/g}
   $$

# Done in Data Preparation by merging (environmental, taxa, chemical) tabulars

The chemical data takes most time in data preparing, the other two data sets - environmental features and
taxa relative abundances are clean already.

### The common keys for merging the three

The three tabulars can be merged in site-wise(row) manner by two common keys - `StationID` and `Integrated Code`.
The shared index columns are 'StationID' and 'Integrated Code'. '**StationID**' can be found in 
(environmental, chemical) tabulars, while **Integrated Code** can be found in (taxa, chemical) tabulars.

Therefore, a site can have its three types of data merged by the keys - (StationID, Integrated Code) as following:

$$
\text{environmental} \xleftrightarrow{\text{StationID}} \text{chemical} \xleftrightarrow{\text{Integrated Code}} \text{taxa}
$$

Other sample information such as year, waterbody, etc. will be saved only once by complementary searching from the three tabulars, because a site might miss its sample information in one tabular but can be found in the other tabulars.

### The union of sites in all three tabulars (shape of the merged dataset)

Initially, the three tabulars were in the following shapes and units:

<div align="center">

<table style="margin-left: center; margin-right: center; text-align: center;">
    <thead>
        <tr><th>data type</th><th>shape</th><th>unit</th></tr>
    </thead>
    <tbody>
        <tr><td>environmental</td><td>(323, 5 + 6)</td><td>unit array (6, 1)</td></tr>
        <tr><td>taxa</td><td>(311, 5 + 16)</td><td>octave format</td></tr>
        <tr><td>chemical</td><td>(322, 6 + 16)</td><td>unit array (16, 1)</td></tr>
    </tbody>
</table>

</div>

Each row in each tabular can merge with the rows in the other two tabulars by the common keys, but not all rows can find their matches in the other two tabulars. The union of sites that can be found in all three tabulars are the
rows having complete data in all three types. After merging and reviewing, there are 310 sites having complete data in all three types with shape:

<div align="center">

<table style="margin-left: center; margin-right: center; text-align: center;">
  <thead>
    <tr><th>data type</th><th>shape</th></tr>
  </thead>
  <tbody>
    <tr><td>(environmental, chemical, taxa)</td><td>(310, 5 + 6 + 16 + 16)</td></tr>
  </tbody>
</table>

</div>

**Conclusions**:


Complete data of all three types are clean and ready to use, the units of the three types are different and need to explain before analysis. The taxa values were in **log2-transformed relative abundace with a pseudo-count** or
in **octave-scale transformation of relative abundance** (Gauch, 1982) originally.
Its formula is attached below, where $p_ij$ is the proportion of taxa $j$ at site $i$ and the $y_{ij}$ is the
octave-scale transformed value of taxa $j$ at site $i$. 
This octave default values support to infer the relative abundances ($p_{ij}$) of taxa $j$ at site $i$, 
but it is impossible to infer the raw absolute abundances.

$$
y_{ij} = log_{2} [100 (p_{ij} + 0.01)]
$$


Let us call them mathematically as matrices with the following notations:
| Data Matrix Source | Symbol | Dimension | Units |
|---|---|---|---|
| Environmental data | $\mathbf{E}$ | $310 \times 6$ | unit array $(6, 1)$ |
| Chemical data (material composition) | $\mathbf{M}$ | $310 \times 16$ | unit array $(16, 1)$ |
| Taxa data (relative abundances) | $\mathbf{T}$ | $310 \times 16$ | octave format |

where $\mathbf{E}$ denotes environmental data, $\mathbf{M}$ denotes chemical data, and $\mathbf{T}$ denotes taxa data.